In [1]:
# ============================================================
# ──────────────────────────────────────────────
# CELL 1 — Install Dependencies
# ──────────────────────────────────────────────
!pip install wikipedia-api nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 3.0 MB/s eta 0:00:00


In [2]:
# ──────────────────────────────────────────────
# CELL 2 — Imports & NLTK Downloads
# ──────────────────────────────────────────────
import wikipediaapi
import nltk
import re
import random
from collections import defaultdict, Counter

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

print("✅ All imports successful")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


✅ All imports successful


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
# ──────────────────────────────────────────────
# CELL 3 — Download Wikipedia Corpus (10,000+ words)
# ──────────────────────────────────────────────
wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='Arya-5012/1.0'
)

# Multiple articles to ensure 10,000+ words
topics = [
    "Artificial intelligence",
    "Machine learning",
    "Natural language processing",
    "Deep learning",
    "Neural network"
]

corpus_raw = ""

for topic in topics:
    page = wiki.page(topic)
    if page.exists():
        corpus_raw += " " + page.text
        print(f"✅ Fetched: '{topic}' — {len(page.text.split())} words")

print(f"\n📦 Total Corpus Size: {len(corpus_raw.split())} words")

✅ Fetched: 'Artificial intelligence' — 12769 words
✅ Fetched: 'Machine learning' — 8495 words
✅ Fetched: 'Natural language processing' — 4498 words
✅ Fetched: 'Deep learning' — 8208 words
✅ Fetched: 'Neural network' — 614 words

📦 Total Corpus Size: 34584 words


In [4]:
# ──────────────────────────────────────────────
# CELL 4 — Text Preprocessing & Tokenization
# ──────────────────────────────────────────────
def preprocess(text):
    # Lowercase
    text = text.lower()
    # Remove special characters, digits, extra spaces
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Clean corpus
corpus_clean = preprocess(corpus_raw)

# Tokenize into words
tokens = nltk.word_tokenize(corpus_clean)

print(f"📌 Total Tokens After Cleaning : {len(tokens)}")
print(f"📌 Unique Tokens (Vocab Size)  : {len(set(tokens))}")
print(f"\n🔍 Sample Tokens (first 20):")
print(tokens[:20])

📌 Total Tokens After Cleaning : 34095
📌 Unique Tokens (Vocab Size)  : 5492

🔍 Sample Tokens (first 20):
['artificial', 'intelligence', 'ai', 'is', 'the', 'capability', 'of', 'computational', 'systems', 'to', 'perform', 'tasks', 'typically', 'associated', 'with', 'human', 'intelligence', 'such', 'as', 'learning']


In [5]:
# ──────────────────────────────────────────────
# CELL 5 — Build Unigram Model
# ──────────────────────────────────────────────
def build_unigram_model(tokens):
    """
    Unigram model: P(word) = count(word) / total_words
    Every word is independent — no context considered.
    """
    total = len(tokens)
    freq  = Counter(tokens)
    probs = {word: count / total for word, count in freq.items()}
    return probs, freq

unigram_probs, unigram_freq = build_unigram_model(tokens)

# Show top 15 most probable words
top15 = sorted(unigram_probs.items(), key=lambda x: x[1], reverse=True)[:15]

print("📊 Top 15 Unigram Probabilities:")
print(f"{'Word':<20} {'Probability':<15} {'Count'}")
print("-" * 45)
for word, prob in top15:
    print(f"{word:<20} {prob:<15.6f} {unigram_freq[word]}")

📊 Top 15 Unigram Probabilities:
Word                 Probability     Count
---------------------------------------------
the                  0.047690        1626
of                   0.031882        1087
and                  0.028597        975
to                   0.025077        855
a                    0.024461        834
in                   0.023024        785
learning             0.012553        428
that                 0.012553        428
is                   0.012348        421
as                   0.009239        315
for                  0.009151        312
by                   0.008036        274
ai                   0.007420        253
are                  0.006687        228
on                   0.006687        228


In [6]:
# ──────────────────────────────────────────────
# CELL 6 — Build Bigram Model
# ──────────────────────────────────────────────
def build_bigram_model(tokens):
    """
    Bigram model: P(word_n | word_n-1) = count(word_n-1, word_n) / count(word_n-1)
    Uses previous word as context.
    """
    bigram_freq   = defaultdict(Counter)
    bigram_probs  = defaultdict(dict)

    # Count bigrams
    for w1, w2 in zip(tokens[:-1], tokens[1:]):
        bigram_freq[w1][w2] += 1

    # Convert to probabilities
    for w1, next_words in bigram_freq.items():
        total = sum(next_words.values())
        for w2, count in next_words.items():
            bigram_probs[w1][w2] = count / total

    return bigram_probs, bigram_freq

bigram_probs, bigram_freq = build_bigram_model(tokens)

# Show sample bigrams for the word "neural"
sample_word = "neural"
if sample_word in bigram_probs:
    top_next = sorted(bigram_probs[sample_word].items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"📊 Top 5 words that follow '{sample_word}':")
    print(f"{'Next Word':<20} {'P(word | \"{sample_word}\")'}")
    print("-" * 40)
    for word, prob in top_next:
        print(f"{word:<20} {prob:.6f}")

📊 Top 5 words that follow 'neural':
Next Word            P(word | "{sample_word}")
----------------------------------------
networks             0.627219
network              0.213018
history              0.017751
nets                 0.017751
synapses             0.011834


In [7]:
# ──────────────────────────────────────────────
# CELL 7 — Sentence Prediction (Length = 15)
# ──────────────────────────────────────────────
def predict_unigram(start_word, length=15):
    """
    Generates sentence using Unigram model.
    Each next word is sampled purely from unigram distribution
    — completely ignores context (dumb but baseline).
    """
    vocab  = list(unigram_probs.keys())
    probs  = list(unigram_probs.values())

    sentence = [start_word]
    for _ in range(length - 1):
        next_word = random.choices(vocab, weights=probs, k=1)[0]
        sentence.append(next_word)

    return ' '.join(sentence)


def predict_bigram(start_word, length=15):
    """
    Generates sentence using Bigram model.
    Each next word is sampled from P(word | previous_word).
    Falls back to unigram if the word has no bigram context.
    """
    sentence    = [start_word]
    current     = start_word

    for _ in range(length - 1):
        if current in bigram_probs:
            next_words  = list(bigram_probs[current].keys())
            weights     = list(bigram_probs[current].values())
            current     = random.choices(next_words, weights=weights, k=1)[0]
        else:
            # Fallback: unigram sampling
            current = random.choices(
                list(unigram_probs.keys()),
                weights=list(unigram_probs.values()), k=1
            )[0]
        sentence.append(current)

    return ' '.join(sentence)


# ── Run Predictions ──
starting_word = "learning"   # ← Change this to any word you like

print(f"🚀 Starting Word: '{starting_word}'\n")
print("=" * 60)
print("📌 UNIGRAM Prediction (no context):")
print(predict_unigram(starting_word, length=15))
print()
print("📌 BIGRAM Prediction (uses previous word as context):")
print(predict_bigram(starting_word, length=15))
print("=" * 60)

🚀 Starting Word: 'learning'

📌 UNIGRAM Prediction (no context):
learning new hidden solve vulnerabilities of in or had in fox specificity single knowledge modern

📌 BIGRAM Prediction (uses previous word as context):
learning uses english this is considered granting electronic personhood credentials as far term for ai


In [8]:
# ──────────────────────────────────────────────
# CELL 8 — Compare Multiple Starting Words
# ──────────────────────────────────────────────
test_words = ["machine", "language", "data", "network", "model"]

print("🔬 Sentence Predictions — Unigram vs Bigram\n")
print("=" * 70)

for word in test_words:
    if word not in unigram_probs:
        print(f"⚠️  '{word}' not in vocabulary. Skipping.\n")
        continue

    print(f"▶ Starting word: '{word}'")
    print(f"  [UNIGRAM] {predict_unigram(word, 15)}")
    print(f"  [BIGRAM ] {predict_bigram(word, 15)}")
    print()

🔬 Sentence Predictions — Unigram vs Bigram

▶ Starting word: 'machine'
  [UNIGRAM] machine from unethical the vector of machines belongs signal human models representations perception popular list
  [BIGRAM ] machine learning requires consideration by the raw input used to humanity may possibly the book

▶ Starting word: 'language'
  [UNIGRAM] language adequate learning are ethics the the etc in breakthrough slower sutskever smartphone artificial arrangement
  [BIGRAM ] language processing tasks from biology such as what the late s generative methods based on

▶ Starting word: 'data'
  [UNIGRAM] data achieving it logistics problems safety statistical bioinformatics to in used one fields structures on
  [BIGRAM ] data some other applications automatic speech recognition be thought it texttospeech given and the more

▶ Starting word: 'network'
  [UNIGRAM] network argument of geoffrey problems deep rulebased the judgments are context networks refers in with
  [BIGRAM ] network is dependen

In [9]:
#CellJob3Pulls 5 Wikipedia articles → guaranteed 10K+ word corpus4Cleans text + tokenizes using NLTK5Unigram — P(word) = count / total6Bigram — P(word₂ | word₁) = count(w1,w2) / count(w1)7Predicts 15-word sentence for both models from a start word8Runs comparison across multiple starting words
#The key insight your professor wants to see: Bigram output will read noticeably more natural than Unigram — because context matters.
